# 💬 Mini-Chat 2.0 - Homework Guided Exercise

Build a small, stateful local chatbot while staying focused on the LLM foundations from Module 2.

This notebook is a standalone extension of the final mini-chat demonstration in `short_introduction_to_LLMs.ipynb`. The classroom notebook introduced tokenization, chat templates, generation, and a deliberately simple low-level chatbot. This exercise keeps those ideas, but adds just enough application logic to make the chatbot more usable.

## Why This Exercise Matters

A model call does not automatically create a conversation. The application must decide:

- which messages belong in the next prompt,
- how generation settings affect behavior,
- what to do when the history becomes too long,
- and how user commands change the application state.

The purpose is **not to learn PyTorch**. Model loading, tokenization, chat-template handling, and streaming generation are supplied. The required work uses ordinary Python functions, lists, dictionaries, loops, and conditionals.

## Assignment Scope

### Required core

Your completed core application should support:

1. Multi-turn conversation history using `system`, `user`, and `assistant` messages.
2. Three supplied generation presets: `precise`, `balanced`, and `creative`.
3. The commands `/reset`, `/preset <name>`, `/help`, and `/exit`.
4. A fixed input-token budget.
5. Removal of the oldest **complete user-assistant turns** when the budget is exceeded.
6. A working interactive loop.

The required student-owned work is concentrated in three functions, one per required task.


### Optional tasks


The notebook also includes two independent optional tasks after the required application is complete. You may skip them without affecting the core chatbot:

- changing the system message,
- saving the current conversation as JSON.


### Expected workload

A reasonable target is:

- **Required core:** approximately 2–4 hours, depending on Python proficiency.
- **One optional task:** approximately 30–60 additional minutes.


## Architecture

```text
user input
    │
    ├── command ───────────────> update application state
    │
    └── ordinary message
            │
            ├── create candidate history
            ├── trim oldest complete turns if needed
            ├── apply the model's chat template
            ├── generate and stream the answer
            └── store the completed user-assistant turn
```

A candidate history is used while generating. The application updates its real history only after a response has been produced successfully.

## 1. Environment Setup

Run the installation cell once.

Qwen3 requires a recent Transformers version. CPU inference is expected to be slower than GPU inference.

In [ ]:
# !pip install "transformers>=4.51" accelerate

In [ ]:
from typing import Any, Iterator
from threading import Thread

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

## 2. Models and Configuration

The notebook automatically uses the classroom model on CUDA. CPU-only students use the smaller Qwen3 model, with SmolLM2 available as a lower-memory alternative.

Change only the configuration values when a different path is needed. The rest of the notebook uses the same Transformers interface for every model.

In [ ]:
GPU_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
CPU_MODEL_ID = "Qwen/Qwen3-0.6B"
LOW_MEMORY_CPU_MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

# Leave as None for automatic detection. Set to "cuda" or "cpu" to override it.
FORCE_DEVICE: str | None = None

# Enable this when Qwen3 is too demanding for the available CPU or RAM.
USE_LOW_MEMORY_CPU_MODEL = False

# "auto" normally works well. Use torch.float32 as a compatibility fallback.
CPU_DTYPE: str | torch.dtype = "auto"

DEFAULT_SYSTEM_MESSAGE = (
    "You are a helpful, concise assistant. "
    "When you are uncertain, say so instead of inventing facts."
)

DEFAULT_PRESET_NAME = "balanced"

# Deliberately small enough that trimming can be observed during testing.
MAX_INPUT_TOKENS = 1_000

### Generation presets

The presets are supplied. Students use them; they do not have to design them from scratch.

- `precise` favors stable output.
- `balanced` allows controlled variation.
- `creative` allows more diverse token choices.

A higher temperature does not make the model smarter. It changes how candidate tokens are sampled.

In [ ]:
GENERATION_PRESETS: dict[str, dict[str, Any]] = {
    "precise": {
        "max_new_tokens": 120,
        "do_sample": False,
        "repetition_penalty": 1.05,
    },
    "balanced": {
        "max_new_tokens": 160,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "repetition_penalty": 1.08,
    },
    "creative": {
        "max_new_tokens": 180,
        "do_sample": True,
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 50,
        "repetition_penalty": 1.10,
    },
}

for name, settings in GENERATION_PRESETS.items():
    print(f"{name:>8}: {settings}")

## 3. Supplied Model Loading

The following functions isolate hardware and model selection from the exercise logic. Read them, but do not treat them as student implementation tasks.

In [ ]:
def select_device_type() -> str:
    
    if FORCE_DEVICE is not None:
        requested = FORCE_DEVICE.lower()
        if requested not in {"cuda", "cpu"}:
            raise ValueError("FORCE_DEVICE must be None, 'cuda', or 'cpu'.")
        if requested == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested, but PyTorch cannot access a CUDA GPU.")
        return requested

    return "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def select_model_id(device_type: str) -> str:
    
    if device_type == "cuda":
        return GPU_MODEL_ID
    
    if USE_LOW_MEMORY_CPU_MODEL:
        return LOW_MEMORY_CPU_MODEL_ID
    
    return CPU_MODEL_ID

In [ ]:
def identify_model_family(model_id: str) -> str:
    
    lowered = model_id.lower()
    if "qwen3" in lowered:
        return "qwen3"
    if "smollm2" in lowered:
        return "smollm2"
    if "llama-3.2" in lowered:
        return "llama3"
    
    return "other"

In [ ]:
def load_runtime() -> dict[str, Any]:
    
    device_type = select_device_type()
    model_id = select_model_id(device_type)

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    if device_type == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=CPU_DTYPE,
            device_map="cpu",
            low_cpu_mem_usage=True,
        )

    model.eval()
    input_device = next(model.parameters()).device

    return {
        "model_id": model_id,
        "model_family": identify_model_family(model_id),
        "model": model,
        "tokenizer": tokenizer,
        "device_type": device_type,
        "device": input_device,
    }

### Load the runtime

The first run downloads the selected model. CPU loading and generation may take noticeably longer.

In [ ]:
runtime = load_runtime()

print("Model:", runtime["model_id"])
print("Device:", runtime["device"])

## 4. Supplied Chat Formatting and Token Counting

The model does not receive the Python list directly. Its tokenizer converts the list of role-tagged messages into the model-specific chat format.

> **Note**: Qwen3 thinking mode is disabled so that the exercise stays focused on ordinary chat history and generation.

In [ ]:
Message = dict[str, str]

In [ ]:
def tokenize_chat(
    messages: list[Message],
    runtime: dict[str, Any],
):
    template_kwargs: dict[str, Any] = {
        "tokenize": True,
        "add_generation_prompt": True,
        "return_dict": True,
        "return_tensors": "pt",
    }

    if runtime["model_family"] == "qwen3":
        template_kwargs["enable_thinking"] = False

    return runtime["tokenizer"].apply_chat_template(
        messages,
        **template_kwargs,
    )


def count_prompt_tokens(
    messages: list[Message],
    runtime: dict[str, Any],
) -> int:
    tokenized = tokenize_chat(messages, runtime)
    return int(tokenized["input_ids"].shape[-1])

### Inspect one formatted conversation

In [ ]:
demo_messages = [
    {"role": "system", "content": DEFAULT_SYSTEM_MESSAGE},
    {"role": "user", "content": "Explain tokenization in one sentence."},
]

demo_tokens = tokenize_chat(demo_messages, runtime)
print("Prompt tokens:", demo_tokens["input_ids"].shape[-1])
print(runtime["tokenizer"].decode(demo_tokens["input_ids"][0]))

## 5. Supplied Streaming Generation

Streaming makes the chatbot feel responsive, but it is framework plumbing rather than the learning target. The helper is supplied as one isolated function.

In [ ]:
def stream_model_response(
    messages: list[Message],
    preset: dict[str, Any],
    runtime: dict[str, Any],
) -> Iterator[str]:
    
    tokenized = tokenize_chat(messages, runtime).to(runtime["device"])
    tokenizer = runtime["tokenizer"]
    model = runtime["model"]

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs: dict[str, Any] = {
        **tokenized,
        **preset,
        "streamer": streamer,
        "pad_token_id": tokenizer.pad_token_id,
    }

    worker = Thread(
        target=model.generate,
        kwargs=generation_kwargs,
        daemon=True,
    )
    worker.start()

    for text_piece in streamer:
        yield text_piece

    worker.join()

In [ ]:
def generate_assistant_response(
    messages: list[Message],
    preset: dict[str, Any],
    runtime: dict[str, Any],
) -> str:
    
    chunks: list[str] = []

    print("🤖 Assistant: ", end="", flush=True)

    for chunk in stream_model_response(messages, preset, runtime):
        chunks.append(chunk)
        print(chunk, end="", flush=True)

    print()
    
    return "".join(chunks).strip()

## 6. 🫵 Task 1 - Conversation State

The application state is intentionally small:

```python
{
    "history": [...],
    "preset_name": "balanced",
}
```

The history always begins with a system message. The active preset is application state, but it is not part of the conversation sent to the model.

Implement `new_app_state()` so that each new chat receives a fresh history list. Do not reuse one mutable list between sessions.

### Your implementation


In [ ]:
def new_app_state(
    system_message: str = DEFAULT_SYSTEM_MESSAGE,
) -> dict[str, Any]:
    """
    Create and return the initial application state for a new chat.

    Requirements:
    - reject an empty system message,
    - begin history with exactly one system message,
    - use DEFAULT_PRESET_NAME as the active preset,
    - create a fresh history list for every call.
    """

    # TODO: implement Task 1.
    raise NotImplementedError("Task 1: implement new_app_state")


### Sanity check

In [ ]:
state_a = new_app_state()
state_b = new_app_state()

state_a["history"].append({"role": "user", "content": "Hello"})

assert len(state_a["history"]) == 2
assert len(state_b["history"]) == 1
assert state_b["preset_name"] == "balanced"

print("Conversation-state check passed.")

## 7. 🫵 Task 2 - Enforce the Token Budget

The trimming function is called **after** the latest user message has been added and **before** generation.

At that moment, the history follows this shape:

```text
system,
user, assistant,
user, assistant,
...
latest user
```

When the prompt is too long:

1. Preserve the system message.
2. Preserve the latest user message.
3. Remove the oldest `user, assistant` pair.
4. Repeat only while necessary.
5. Raise an error when the system message and latest user message cannot fit by themselves.

The function mutates the candidate history and returns the number of removed turns.

### Your implementation


In [ ]:
def trim_history_to_budget(
    history: list[Message],
    max_input_tokens: int,
    runtime: dict[str, Any],
) -> int:
    """
    Mutate `history` until its rendered prompt fits within the token budget.

    Return the number of complete user-assistant turns that were removed.
    Raise ValueError when the budget is invalid or when the protected
    messages cannot fit by themselves.
    """

    # TODO: implement Task 2.
    #
    # Useful facts:
    # - count_prompt_tokens(history, runtime) measures the rendered prompt.
    # - history[0] is the protected system message.
    # - the oldest removable complete turn starts at history[1].
    raise NotImplementedError("Task 2: implement trim_history_to_budget")


### Test the trimming policy without generating text

In [ ]:
latest_user = {"role": "user", "content": "What topic were we discussing?"}
minimum_history = [
    {"role": "system", "content": DEFAULT_SYSTEM_MESSAGE},
    latest_user,
]

# Enough room for the system message and latest user message, but not for
# the deliberately long old turns below.
test_budget = count_prompt_tokens(minimum_history, runtime) + 8

test_history = [
    {"role": "system", "content": DEFAULT_SYSTEM_MESSAGE},
    {"role": "user", "content": "Old question A. " * 20},
    {"role": "assistant", "content": "Old answer A. " * 20},
    {"role": "user", "content": "Old question B. " * 20},
    {"role": "assistant", "content": "Old answer B. " * 20},
    latest_user.copy(),
]

removed = trim_history_to_budget(test_history, test_budget, runtime)

assert test_history[0]["role"] == "system"
assert test_history[-1] == latest_user
assert removed == 2
assert count_prompt_tokens(test_history, runtime) <= test_budget

print("Removed turns:", removed)
print("Remaining roles:", [message["role"] for message in test_history])

## 8. 🫵 Task 3 - Core Command Handling

The required command handler recognizes only a small command set:

- `/help` prints the supplied help text.
- `/reset` clears conversation turns but preserves the current system message and preset.
- `/preset <name>` changes the active generation preset.
- `/exit` asks the interactive loop to stop.

For ordinary text, the handler reports that it did not handle the input.

The two Boolean return values mean:

```python
handled, should_exit
```

In [ ]:
HELP_TEXT = """
Commands:
  /help                 show this message
  /reset                clear the conversation history
  /preset <name>        choose precise, balanced, or creative
  /exit                 finish the chat
"""

### Your implementation


In [ ]:
def handle_command(
    user_input: str,
    state: dict[str, Any],
) -> tuple[bool, bool]:
    """
    Handle the required Mini-Chat commands.

    Return:
        (handled, should_exit)

    Ordinary text must return (False, False).
    Any recognized or unknown slash command counts as handled.
    """

    stripped = user_input.strip()

    if not stripped.startswith("/"):
        return False, False

    command, _, argument = stripped.partition(" ")
    command = command.lower()
    argument = argument.strip().lower()

    # TODO: implement /help, /reset, /preset, /exit,
    # and the unknown-command behavior described above.
    raise NotImplementedError("Task 3: implement handle_command")


### Command checks

In [ ]:
command_state = new_app_state()
command_state["history"].extend([
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi"},
])

assert handle_command("ordinary text", command_state) == (False, False)
assert handle_command("/preset creative", command_state) == (True, False)
assert command_state["preset_name"] == "creative"
assert handle_command("/reset", command_state) == (True, False)
assert len(command_state["history"]) == 1
assert command_state["preset_name"] == "creative"
assert handle_command("/exit", command_state) == (True, True)

print("Command checks passed.")

## 9. Assemble One Conversation Turn

This function ties the three tasks together.

Notice that it first creates a **candidate history**. The real application history changes only after the model has produced an answer. This avoids leaving an unmatched user message in the stored conversation if generation fails.

In [ ]:
def process_user_message(
    user_input: str,
    state: dict[str, Any],
    runtime: dict[str, Any],
) -> int:
    
    user_input = user_input.strip()
    if not user_input:
        raise ValueError("The user message cannot be empty.")

    candidate_history = [
        *state["history"],
        {"role": "user", "content": user_input},
    ]

    removed_turns = trim_history_to_budget(
        candidate_history,
        MAX_INPUT_TOKENS,
        runtime,
    )

    preset = GENERATION_PRESETS[state["preset_name"]]
    assistant_text = generate_assistant_response(
        candidate_history,
        preset,
        runtime,
    )

    if not assistant_text:
        raise RuntimeError("The model produced an empty response.")

    candidate_history.append({
        "role": "assistant",
        "content": assistant_text,
    })
    state["history"] = candidate_history

    return removed_turns

## 10. Supplied Interactive Loop

The loop is supplied. It delegates application behavior to the functions implemented above.

In [ ]:
def run_mini_chat(
    runtime: dict[str, Any],
    state: dict[str, Any] | None = None,
) -> dict[str, Any]:
    state = state or new_app_state()

    print("💬 Mini-Chat 2.0")
    print("Model:", runtime["model_id"])
    print("Use /help for commands. Use /exit to finish.\n")

    while True:
        
        try:
            user_input = input("🧑 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Bye!")
            break

        if not user_input:
            continue

        handled, should_exit = handle_command(user_input, state)

        if should_exit:
            print("👋 Bye!")
            break

        if handled:
            print()
            continue

        try:
            removed_turns = process_user_message(user_input, state, runtime)
            if removed_turns:
                print(f"   [Removed {removed_turns} old turn(s) to fit the token budget.]", end="\n\n")
            else:
                print()
        except Exception as exc:
            print(f"⚠️ {type(exc).__name__}: {exc}\n")

    return state

### Run Mini-Chat 2.0

In [ ]:
app_state = run_mini_chat(runtime)

## 11. Required Behavioral Checks

These checks verify deterministic application behavior. They deliberately do not grade the wording of model-generated answers.

In [ ]:
def run_behavioral_checks() -> None:
    # A new state contains one system message and the default preset.
    state = new_app_state("Test system")
    assert state["history"] == [
        {"role": "system", "content": "Test system"},
    ]
    assert state["preset_name"] == "balanced"

    # Preset selection changes application state.
    handled, should_exit = handle_command("/preset creative", state)
    assert handled and not should_exit
    assert state["preset_name"] == "creative"

    # Reset removes turns but preserves the system message and preset.
    state["history"].extend([
        {"role": "user", "content": "Hello"},
        {"role": "assistant", "content": "Hi"},
    ])
    handle_command("/reset", state)
    assert state["history"] == [
        {"role": "system", "content": "Test system"},
    ]
    assert state["preset_name"] == "creative"

    # The trimming check from Task 2 already verifies whole-turn removal.
    print("All required behavioral checks passed.")

In [ ]:
run_behavioral_checks()

## 12. Suggested Manual Test Session

Try a short sequence that reveals the application state:

1. Tell the assistant a made-up name and one preference.
2. Ask a follow-up question that depends on that information.
3. Switch to `/preset creative` and request three unusual project names.
4. Use `/reset`, then ask what name you gave earlier.
5. Send several deliberately long messages and watch for the trimming notice.
6. Finish with `/exit`.

The model may occasionally claim to remember something after reset. That is model behavior, not proof that the old history was still sent. Inspect `app_state["history"]` when in doubt.

## 13. 🫵 Optional Task 4 - Change the System Message

Add support for inspecting and changing the system message while preserving the rest of the conversation.

Implement `set_system_message()` and `handle_system_command()` according to these requirements:

1. Strip surrounding whitespace from a new system message.
2. Reject an empty system message with `ValueError`.
3. Replace the first history item instead of appending another system message.
4. Preserve all existing `user` and `assistant` messages.
5. `/system` with no argument prints the current system message.
6. `/system <message>` updates the system message.
7. Return `True` when the input is a `/system` command and `False` otherwise.

This task is intentionally independent of the required command handler. After it works, it can be wired into the interactive loop as a further extension.


### Your implementation


In [ ]:
def set_system_message(
    state: dict[str, Any],
    new_message: str,
) -> None:
    """
    Replace the system message while preserving all conversation turns.
    """

    # TODO: implement Optional Task 4A.
    raise NotImplementedError("Optional Task 4A: implement set_system_message")


def handle_system_command(
    user_input: str,
    state: dict[str, Any],
) -> bool:
    """
    Inspect or update the system message when `user_input` is a /system command.

    Return True when the input was handled as /system and False otherwise.
    """

    # TODO: implement Optional Task 4B.
    raise NotImplementedError("Optional Task 4B: implement handle_system_command")

### Optional-task checks


In [ ]:
optional_state = new_app_state()
optional_state["history"].extend([
    {"role": "user", "content": "My favorite color is green."},
    {"role": "assistant", "content": "I will remember that."},
])

preserved_turns = [message.copy() for message in optional_state["history"][1:]]

assert handle_system_command("ordinary text", optional_state) is False
assert handle_system_command(
    "/system You answer like a friendly science-fiction computer.",
    optional_state,
) is True

assert optional_state["history"][0] == {
    "role": "system",
    "content": "You answer like a friendly science-fiction computer.",
}
assert optional_state["history"][1:] == preserved_turns
assert handle_system_command("/system", optional_state) is True

try:
    set_system_message(optional_state, "   ")
except ValueError:
    pass
else:
    raise AssertionError("An empty system message should raise ValueError.")

print("Optional Task 4 checks passed.")


## 14. 🫵 Optional Task 5 - Save the Conversation

Add simple JSON persistence so a conversation can be inspected or reused later.

Implement `save_conversation()` according to these requirements:

1. Accept the application state, runtime information, and an optional filename.
2. Ensure the output filename ends in `.json`.
3. Save only clear, serializable data: application name, model ID, active preset, and message history.
4. Write UTF-8 JSON with readable indentation and preserve non-ASCII text.
5. Return the resulting `Path` object.

Do not serialize the model, tokenizer, tensors, or other runtime objects. This task is about application data persistence, not model persistence.


### Your implementation


In [ ]:
import json
from pathlib import Path

In [ ]:
def save_conversation(
    state: dict[str, Any],
    runtime: dict[str, Any],
    filename: str = "mini_chat_transcript.json",
) -> Path:
    """
    Save the current application data as readable UTF-8 JSON.

    Do not serialize the model, tokenizer, tensors, or other runtime objects.
    Return the final output path.
    """

    # TODO: implement Optional Task 5.
    raise NotImplementedError("Optional Task 5: implement save_conversation")

### Optional-task checks


In [ ]:
from tempfile import TemporaryDirectory


with TemporaryDirectory() as temporary_directory:
    test_state = new_app_state("Answer briefly.")
    test_state["history"].extend([
        {"role": "user", "content": "Remember the word שלום."},
        {"role": "assistant", "content": "I will remember it."},
    ])
    test_state["preset_name"] = "precise"
    test_runtime = {"model_id": "test-model"}

    output_file = save_conversation(
        test_state,
        test_runtime,
        Path(temporary_directory) / "saved_chat",
    )

    assert output_file.suffix == ".json"
    assert output_file.exists()

    loaded_payload = json.loads(output_file.read_text(encoding="utf-8"))
    assert loaded_payload["application"] == "Mini-Chat 2.0"
    assert loaded_payload["model"] == "test-model"
    assert loaded_payload["preset"] == "precise"
    assert loaded_payload["messages"] == test_state["history"]
    assert "שלום" in output_file.read_text(encoding="utf-8")

print("Optional Task 5 checks passed.")

### Apply the optional task to your chat


In [ ]:
# Uncomment after running the chatbot.
# transcript_path = save_conversation(app_state, runtime)
# print("Saved to:", transcript_path.resolve())


## 15. Reflection Questions

1. Why does the model need the full selected history on every turn?
2. Why must trimming remove a complete user-assistant pair rather than arbitrary messages?
3. What behavior changed when you switched presets?
4. What information is lost when old turns are deleted?
5. Why might a larger application summarize old turns instead of deleting them?
6. Which parts of this notebook are LLM-specific, and which are ordinary application logic?

## Takeaways

Mini-Chat 2.0 demonstrates a small set of foundational AI-engineering ideas:

- Chat history is application state.
- The tokenizer and chat template determine the actual model input.
- Generation settings are part of application behavior.
- Context limits require an explicit history policy.
- Model infrastructure can be separated from ordinary Python logic.

Later modules can add structured outputs, retrieval, reliability controls, observability, persistence, and deployment without changing these foundations.

<hr/>